In [1]:
import numpy as np
import torch
import torch.nn as nn

In [1]:
# create embeddings. 

dic = {}
original = set()
idx = 0

with open("output.txt", "r") as f:
    for token in f.read().split():
        if token not in original:
            original.add(token)
            dic[token] = idx
            idx += 1

NUM_UNIQUE_WORDS = len(dic)

In [2]:
# converting the dataset of words into embeddings.

dataset = []

with open("output.txt", "r") as f:
    for line in f.readlines():
        sentence = []
        for word in str(line).split():
            sentence.append(dic[word])
        
        dataset.append(sentence)

In [ ]:
dataset

[[0, 1, 2],
 [0, 1, 3],
 [4, 5, 6],
 [7, 8],
 [9, 10],
 [11],
 [12, 13, 14, 15, 16, 17, 18, 19, 17, 20],
 [21],
 [22, 23, 24, 25, 19, 26, 27, 28, 29, 30, 31, 32, 27, 33, 25, 19, 17, 34],
 [35, 36, 37, 38, 39, 40],
 [41, 42, 43],
 [44, 45, 46],
 [4, 47, 48, 49, 17, 50, 45, 51, 52],
 [53, 54, 55, 4, 56, 17, 57, 58, 59, 60, 61, 62, 63],
 [64, 65, 66, 67, 68, 22, 69, 70, 71],
 [44, 72],
 [1, 27, 73, 17, 30, 72],
 [74],
 [75, 76, 77, 78, 27, 79, 80, 81, 14, 15, 82, 83, 84, 85, 86, 87],
 [24, 88, 89, 30, 90],
 [91],
 [75, 92, 93, 27, 56, 17, 94],
 [95, 96],
 [24, 97, 98, 17, 79, 99, 100, 101, 102, 103],
 [4, 104],
 [4, 105, 106, 27, 107, 108, 50, 109, 110, 27, 88, 111, 17, 112, 113],
 [114],
 [115, 116, 117],
 [4, 118, 119, 120, 84, 121, 122],
 [24, 25, 123],
 [124, 83, 125, 126, 99, 127, 128],
 [129],
 [130,
  131,
  132,
  133,
  134,
  135,
  136,
  137,
  138,
  139,
  140,
  141,
  142,
  143,
  144,
  145,
  139,
  146],
 [147, 92, 50, 148, 38, 145, 149, 150, 151, 152],
 [153,
  154,
 

In [ ]:
class GRU(nn.Module): # unidirectional
    def __init__(self, input_size, hidden_size) -> None:
        super().__init__()

        self.sigmoid = nn.Sigmoid()
        self.tanh = nn.Tanh()

        self.restore_fc = nn.Linear(input_size + hidden_size, hidden_size)
        self.update_fc = nn.Linear(input_size + hidden_size, hidden_size)

        self.pseudo_current_hidden_fc = nn.Linear(input_size + hidden_size, hidden_size)

    def forward(self, x, h_prev):

        gate_input = torch.concat([x, h_prev], dim=-1)
        
        restore_gate = self.sigmoid(self.restore_fc(gate_input))
        update_gate = self.sigmoid(self.update_fc(gate_input))

        psuedo_hidden_input = torch.concat([x, restore_gate * h_prev], dim=-1)
        psuedo_hidden = self.tanh(self.pseudo_current_hidden_fc(psuedo_hidden_input))

        h_t = (1 - update_gate) * h_prev + update_gate * psuedo_hidden

        return h_t

In [ ]:
class Encoder(nn.Module):
    def __init__(self, hidden_size, embedding, layers=1, dropout=0) -> None:
        super().__init__()

        self.hidden_size = hidden_size
        self.embedding = embedding
        
        self.gru = nn.GRU(input_size=hidden_size, hidden_size=hidden_size, num_layers=layers, dropout=(0 if layers == 1 else dropout), bidirectional=True)

    
    def forward(self, x, input_seq, input_lens, hidden=None):
        
        embedded = self.embedding(input_seq)

        packed = nn.utils.rnn.pack_padded_sequence(embedded, input_lens)

        outputs, hidden = self.gru(packed, hidden)

        outputs, _ = nn.utils.rnn.pad_packed_sequence(outputs)

        outputs = outputs[:, :, :self.hidden_size] + outputs[:, :, self.hidden_size:]

        return outputs, hidden

In [ ]:
class Seq2Seq(nn.Module):
    def __init__(self) -> None:
        super().__init__()